# **Installing Required Libraries**

In [ ]:
!pip install sentence_transformers

In [ ]:
# pip install python-Levenshtein

In [ ]:
!pip install faiss-cpu

In [ ]:
!pip install --upgrade pandas

In [ ]:
pip install fuzzywuzzy

# **Importing Libraries**

In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
import faiss
import torch
from fuzzywuzzy import process
from fuzzywuzzy import fuzz
import os


# **Loading the dataset**

In [ ]:
# Load the dataset
df = pd.read_csv('medquad.csv', encoding='latin-1')  # Change encoding if necessary
df = df.groupby('question').agg({'answer': lambda x: ' '.join(x.astype(str))}).reset_index()
medical_data = dict(zip(df['question'], df['answer']))


In [ ]:
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
df.head(5)

,question,answer
0,Do you have information about A1C,Summary : A1C is a blood test for type 2 diabe...
1,Do you have information about Acupuncture,Summary : Acupuncture has been practiced in Ch...
2,Do you have information about Adoption,Summary : Adoption brings a child born to othe...
3,Do you have information about Advance Directives,Summary : What kind of medical care would you ...
4,Do you have information about African American...,Summary : Every racial or ethnic group has spe...


# **Load Models for NER and Embeddings**

In [ ]:
# Model loading for NER, embeddings, and summarization
model_name = "dmis-lab/biobert-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## **Setting NER pipeline to use CPU if GPU has issues**

In [ ]:
device = "cpu"
ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, device=device)

# **Sentence Embedding Model**

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# **FAISS Indexing**

In [ ]:
embedding_matrix_path = "embedding_matrix.npy"
index_path = "faiss_index_hnsw.bin"

if not os.path.exists(embedding_matrix_path):
    embeddings = []
    batch_size = 64
    for i in range(0, len(df), batch_size):
        batch = df['question'][i:i + batch_size].tolist()
        batch_embeddings = embedder.encode(batch, convert_to_tensor=True, show_progress_bar=True)
        embeddings.append(batch_embeddings)
    embedding_matrix = torch.cat(embeddings).cpu().detach().numpy()
    np.save(embedding_matrix_path, embedding_matrix)
else:
    embedding_matrix = np.load(embedding_matrix_path)

faiss.normalize_L2(embedding_matrix)
d = embedding_matrix.shape[1]


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

# **Using HNSW Index for Faster Retrievl of answers on large datasets**

In [ ]:
if os.path.exists(index_path):
    index = faiss.read_index(index_path)
else:
    index = faiss.IndexHNSWFlat(d, 32)  # Using HNSW index with 32 neighbors
    index.add(embedding_matrix)
    faiss.write_index(index, index_path)


# **Loading Summarization Model**

In [ ]:
# Summarization model setup with CPU fallback
summarizer_model_name = "facebook/bart-large-cnn"
summarizer_tokenizer = AutoTokenizer.from_pretrained(summarizer_model_name)
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(summarizer_model_name)
summarization_pipeline = pipeline("summarization", model=summarizer_model, tokenizer=summarizer_tokenizer, device=device)


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

# **Adding Abbreviation Dictionary**

In [ ]:
abbreviations = {
    "BP": "blood pressure",
    "DM": "diabetes mellitus",
    "P.A.D.": "peripheral arterial disease",
    "HDL": "high-density lipoprotein",
    "LDL": "low-density lipoprotein",
    "HR": "heart rate",
    "MRI": "magnetic resonance imaging"

}


# **Expanding Abbreviations in Text**

In [ ]:
# Expand abbreviations in context
def expand_abbreviations(text):
    words = text.split()
    expanded_words = [abbreviations.get(word, word) for word in words]
    return ' '.join(expanded_words)

df['question'] = df['question'].apply(expand_abbreviations)
medical_data = dict(zip(df['question'], df['answer']))


***Summarizing Text***

In [ ]:
# Summarize text with enhanced length handling
def summarize_text(text, min_length=100):
    max_length = min(len(text) * 4 // 5, 500)
    try:
        summary = summarization_pipeline(
            text, max_length=max_length, min_length=min_length, do_sample=False
        )
        return summary[0]['summary_text']
    except Exception as e:
        print(f"Summarization error: {e}")
        return text[:max_length]


# **Extracting Entities**

In [ ]:
# Improved entity extraction with error handling
def extract_entities(text):
    try:
        tokens = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        outputs = model(**tokens)
        predictions = torch.argmax(outputs.logits, dim=2)
        entities = [tokenizer.decode([tok]) for pred, tok in zip(predictions[0], tokens['input_ids'][0]) if pred != 0]
        return entities
    except Exception as e:
        print(f"Entity extraction error: {e}")
        return []


# **Calculating Confidence Score**

In [ ]:
# Calculate Confidence Scores
def calculate_confidence_scores(question_embedding, ranked_indices):
    # Get embeddings of ranked answers
    answer_embeddings = embedding_matrix[ranked_indices]

    # Compute cosine similarities
    cosine_similarities = np.dot(answer_embeddings, question_embedding.T).flatten()

    # Normalize to a percentage scale (0-100)
    confidence_scores = (cosine_similarities - cosine_similarities.min()) / (
        cosine_similarities.max() - cosine_similarities.min()
    ) * 100
    return confidence_scores


# **Ranking Answers using FASIS**

In [ ]:
# Enhanced Answer Ranking with FAISS
def rank_answers(question_embedding, top_k=5):
    question_embedding = np.array(question_embedding)  # Ensure it's a numpy array
    question_embedding = question_embedding.reshape(1, -1)  # Ensure it's 2D
    faiss.normalize_L2(question_embedding)
    _, indices = index.search(question_embedding, top_k) # top_k is now an integer, defaulting to 5
    ranked_answers = [(df['answer'][i], i) for i in indices[0]]
    return ranked_answers


# **Getting the final answer after all the required steps**

In [ ]:
# Enhanced answer generation function with Confidence Scores
def generate_answer(question):
    question = expand_abbreviations(question)
    question_embedding = embedder.encode([question], convert_to_tensor=True).cpu().detach().numpy()
    ranked_answers = rank_answers(question_embedding) # Removed embedding_matrix argument

    top_ranked_answers = []
    for answer_text, idx in ranked_answers:
        context = medical_data[df['question'][idx]]
        summarized_answer = summarize_text(context)
        top_ranked_answers.append((summarized_answer, idx))

    # Calculate confidence scores
    confidence_scores = calculate_confidence_scores(question_embedding, [idx for _, idx in ranked_answers])

    if top_ranked_answers:
        main_answer = top_ranked_answers[0][0]
        main_confidence = confidence_scores[0]

        related_answers_text = "\n".join(
            [
                f"- {ans} (Confidence: {conf:.2f}%)"
                for ans, conf in zip([ans for ans, _ in top_ranked_answers][1:], confidence_scores[1:])
            ]
        )

        return (
            f"Answer: {main_answer} (Confidence: {main_confidence:.2f}%)\n\n"
            f"Related answers:\n{related_answers_text}"
        )
    else:
        return "Sorry, I couldn't find a relevant answer for your question."


Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Example query
user_input = input("Enter your question: ")
answer = generate_answer(user_input)
print(answer)


Enter your question: what is high BP


Your max_length is set to 500, but your input_length is only 217. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=108)


Summarization error: index out of range in self
Summarization error: index out of range in self
Summarization error: index out of range in self
Answer: High blood pressure is a common disease in which blood flows through blood vessels (arteries) at higher than normal pressures. What Is Blood Pressure? Blood pressure is the force of blood pushing against the walls of the blood vessels as the heart pumps blood. If your blood pressure rises and stays high over time, its called high blood pressure. High blood pressure is dangerous because it makes the heart work too hard, and the high force of the blood flow can harm arteries and organs such as the (Confidence: 100.00%)

Related answers:
- Changes in Body Functions Researchers continue to study how various changes in normal body functions cause high blood pressure. The key functions affected in high blood pressure include -  kidney fluid and salt balances  -  the renin-angiotensin-aldosterone system   - the sympathetic nervous system activ

In [ ]:
# Example query
user_input = input("Enter your question: ")
answer = generate_answer(user_input)
print(answer)

Enter your question: what is fever


Your max_length is set to 500, but your input_length is only 196. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=98)
Your max_length is set to 500, but your input_length is only 206. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=103)
Your max_length is set to 500, but your input_length is only 201. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=100)
Your max_length is set to 500, but your input_length is only 301. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=15

Answer: A fever is a body temperature that is higher than normal. It is not an illness. Fever is part of your body's defense against infection. Most bacteria and viruses that cause infections do well at the body's normal temperature (98.6 F) A slight fever can make it harder for them to survive. Treatment depends on the cause of your fever. Your health care provider may recommend using over-the-counter medicines such as acetaminophen or ibuprofen. Adults can also take aspirin, but children with fevers should not take aspirin. (Confidence: 100.00%)

Related answers:
- As of January 1, 2009, Q fever infections are reported under distinct reporting categories described in the 2009 Q fever surveillance case definition. More detailed information on the diagnosis, management, and treatment of Q fever is available in other sections of this web site. The general public and healthcare providers should first call 1-800-CDC-INFO (1-800.232-4636) for questions regarding Q fever. If a consultation 

In [ ]:
# Example query
user_input = input("Enter your question: ")
answer = generate_answer(user_input)
print(answer)

Enter your question: what is corona virus


Your max_length is set to 500, but your input_length is only 220. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=110)
Your max_length is set to 500, but your input_length is only 366. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=183)
Your max_length is set to 500, but your input_length is only 221. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=110)


Answer: Coronaviruses are common viruses that most people get some time in their life. They usually cause mild to moderate upper-respiratory illness. Some coronavirus can cause severe illness. There is no vaccine to prevent coronavirus infection. There are no specific treatments. You can relieve symptoms with pain and fever medicines and rest. You may be able to reduce your risk of infection by washing your hands often with soap and water, not touching your eyes, nose, or mouth. (Confidence: 100.00%)

Related answers:
- Coronary heart disease (CHD) is a disease in which a waxy substance called plaque builds up inside the coronary arteries. These arteries supply oxygen-rich blood to your heart muscle. Over time, plaque can harden or rupture (break open) Hardened plaque narrows the coronary artery and reduces the flow of oxygen- rich blood to the heart. Heart failure is a condition in which your heart can't pump enough blood to meet your bodys needs. Without quick treatment, a heart atta

In [ ]:
# Example query
user_input = input("Enter your question: ")
answer = generate_answer(user_input)
print(answer)

Enter your question: what is Glaucoma


Your max_length is set to 500, but your input_length is only 216. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=108)


Summarization error: index out of range in self


Your max_length is set to 500, but your input_length is only 332. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=166)
Your max_length is set to 373, but your input_length is only 106. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=53)
Your max_length is set to 500, but your input_length is only 432. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=216)


Answer: Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops  There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. (Watch the video to  (Confidence: 100.00%)

Related answers:
- Nearly 2.7 million people have glaucoma, a leading cause of blindness in the United States. In addition to age, eye pressure is a risk factor. Another risk factor for optic nerve damage relates to blood pressure. It is important to also make sure that your blood pressure is at a proper level for your body by working with your medical doctor. The level of pressure your optic nerve can tolerate without being damaged is different for each person. T

In [ ]:
# Example query
user_input = input("Enter your question: ")
answer = generate_answer(user_input)
print(answer)

Enter your question: what is AIDS


Your max_length is set to 500, but your input_length is only 246. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=123)
Your max_length is set to 500, but your input_length is only 252. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=126)
Your max_length is set to 500, but your input_length is only 169. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=84)
Your max_length is set to 401, but your input_length is only 108. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=54

Summarization error: index out of range in self
Answer: AIDS stands for acquired immunodeficiency syndrome. It is the most advanced stage of infection with HIV. There is no cure, but there are many medicines to fight both HIV infection and the infections and cancers that come with it. A blood test can tell if you have HIV infection. Your health care provider can perform the test, or call the national referral hotline at 1-800-CDC-INFO (24 hours a day, 1-888-232-6348 - TTY) (Confidence: 100.00%)

Related answers:
- Having HIV/AIDS weakens your body's immune system. This can lead to serious infections that don't often affect healthy people. Tuberculosis and a serious related disease, Mycobacterium avium complex (MAC) are bacterial infections. Viral infections include cytomegalovirus (CMV) and hepatitis C. Fungi cause thrush (candidiasis) and cryptococcal meningitis. Parasites cause crypto (cryptosporidiosis) and toxo (toxoplasmosis) (Confidence: 74.48%)
- Some HIV/AIDS medicines may harm

In [ ]:
# Example query
user_input = input("Enter your question: ")
answer = generate_answer(user_input)
print(answer)

Enter your question: What is (are) High Blood Pressure ?
Summarization error: index out of range in self


Your max_length is set to 500, but your input_length is only 217. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=108)


Summarization error: index out of range in self


Your max_length is set to 500, but your input_length is only 245. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=122)


Answer: High blood pressure is a common disease in which blood flows through blood vessels (arteries) at higher than normal pressures. What Is Blood Pressure? Blood pressure is the force of blood pushing against the walls of the blood vessels as the heart pumps blood. If your blood pressure rises and stays high over time, its called high blood pressure. High blood pressure is dangerous because it makes the heart work too hard, and the high force of the blood flow can harm arteries and organs such as the (Confidence: 100.00%)

Related answers:
- High blood pressure, also called hypertension, is an increase in the amount of force that blood places on blood vessels as it moves through the body. Factors that can increase this force include higher blood volume due to extra fluid in the blood and blood vessels that are narrow, stiff, or clogged. Most people without chronic health conditions have a normal blood pressure if it stays below 120/80. People should talk with their health care provi

In [ ]:
# Example query
user_input = input("Enter your question: ")
answer = generate_answer(user_input)
print(answer)

Enter your question: What is (are) Anxiety Disorders ?


Your max_length is set to 500, but your input_length is only 179. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=89)


Summarization error: index out of range in self


Your max_length is set to 500, but your input_length is only 235. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=117)
Your max_length is set to 500, but your input_length is only 189. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=94)
Your max_length is set to 500, but your input_length is only 157. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=78)


Answer: Occasional anxiety is a normal part of life. You might feel anxious when faced with a problem at work, before taking a test, or making an important decision. However, anxiety disorders involve more than temporary worry or fear. For a person with an anxiety disorder, the anxiety does not go away and can get worse over time. These feelings can interfere with daily activities such as job performance, school work, and relationships. (Watch the video to learn about the types of anxiety disorders. To  (Confidence: 100.00%)

Related answers:
- For millions of people in the United States, the anxiety does not go away. They may have chest pains or nightmares, or even be afraid to leave home. These people have anxiety disorders. Treatment can involve medicines, therapy or both.    NIH: National Institute of Mental Health or both, see www.nhmh.org for more information on anxiety disorders, treatment and support for those suffering from the condition. For confidential support call the Sama

In [ ]:
# Example query
user_input = input("Enter your question: ")
answer = generate_answer(user_input)
print(answer)

Enter your question: What is (are) Diabetes ?
Summarization error: index out of range in self
Summarization error: index out of range in self


Your max_length is set to 500, but your input_length is only 259. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=129)
Your max_length is set to 500, but your input_length is only 323. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=161)


Answer: Too Much Glucose in the Blood Diabetes means your blood glucose (often called blood sugar) is too high. Your blood always has some glucose in it because your body needs glucose for energy to keep you going. But too much glucose in the blood isn't good for your health. Glucose comes from the food you eat and is also made in your liver and muscles. Your blood carries the glucose to all of the cells in your body. Insulin is a chemical (a hormone) made by the pancreas. The pancreas releases insulin  (Confidence: 100.00%)

Related answers:
- Diabetes is a complex group of diseases with a variety of causes. People with diabetes have high blood glucose, also called high blood sugar or hyperglycemia.
                
Diabetes is a disorder of metabolismthe way the body uses digested food for energy. The digestive tract breaks down carbohydratessugars and starches found in many foodsinto glucose, a form of sugar that enters the bloodstream. With the help of the hormone insulin, cells th

In [ ]:
# Example query
user_input = input("Enter your question: ")
answer = generate_answer(user_input)
print(answer)


Enter your question: What is (are) Medicare and Continuing Care ?


Your max_length is set to 500, but your input_length is only 285. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=142)


Summarization error: index out of range in self


Your max_length is set to 500, but your input_length is only 193. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=96)
Your max_length is set to 500, but your input_length is only 168. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=84)
Your max_length is set to 500, but your input_length is only 175. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=87)


Answer: Medicare is a federal health insurance program for people - age 65 and older - under age 65 with certain disabilities who have been receiving Social Security disability benefits for a certain amount of time (24 months in most cases) - of any age who have End-Stage Renal Disease (ESRD), which is permanent kidney failure requiring dialysis or a transplant. age 65 and older under age 65 with certain disabilities who have been receiving Social Security disability benefits for a certain amount of tim (Confidence: 100.00%)

Related answers:
- Medicaid is a state and Federal program that will pay most nursing home costs. Medicaid pays for care for about 7 out of every 10 nursing home residents. For information about Medicaid eligibility, call your state Medical Assistance (Medicaid) Office. Visit http://www.medicare.gov on the web for more information about Medicare. For confidential support call the Samaritans on 08457 90 90 90 or visit a local Samaritans branch, see www.samaritans.o

In [ ]:
# Example query
user_input = input("Enter your question: ")
answer = generate_answer(user_input)
print(answer)


Enter your question: What is (are) Knee Replacement ?


Your max_length is set to 500, but your input_length is only 123. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=61)
Your max_length is set to 500, but your input_length is only 233. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=116)
Your max_length is set to 500, but your input_length is only 432. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=216)


Answer: There are many different types and designs of artificial knees. Most consist of three components: the femoral component, the tibial component, and the patellar component. Joint components may also be attached to your own bone in different ways. Most are cemented with a special joint glue into your existing bone; others rely on a process called biologic fixation to hold them in place. In some cases, surgeons use a combination of cemented and uncemented parts. This is referred to as a hybrid implant. (Confidence: 100.00%)

Related answers:
- Exercises to strengthen the muscles around the knee and improve flexibility. Weight loss, if needed, to reduce the load the knee must bear. Walking aids such as canes to reduce stress on the joint. Shoes inserts to improve the knees alignment. medicines to relieve pain. If you are considering knee replacement, see your doctor for more information. For confidential support call the Samaritans in the UK on 08457 90 90 90, visit a local Samarita

In [ ]:
# Example query
user_input = input("Enter your question: ")
answer = generate_answer(user_input)
print(answer)


Enter your question: what is monkey pox


Your max_length is set to 244, but your input_length is only 70. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=35)
Your max_length is set to 500, but your input_length is only 495. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=247)
Your max_length is set to 500, but your input_length is only 390. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=195)
Your max_length is set to 500, but your input_length is only 300. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=150

Answer: Monkeypox occurs mostly in central and western Africa. Wild rodents and squirrels carry it, but it is called monkeypox because scientists saw it first in lab monkeys. In 2003, it was reported in prairie dogs and humans in the U.S.    The Centers for Disease Control and Prevention has more information on monkeypox at: http://www.cdc.gov/monkeypox/monkey-pox-viral-disease-symptoms-and-diagnosis. (Confidence: 100.00%)

Related answers:
- Pompe disease is caused by mutations in a gene that makes an enzyme called acid alpha-glucosidase (GAA) Normally, the body uses GAA to break down glycogen, a stored form of sugar used for energy. In Pompe disease, mutations in the GAA gene reduce or completely eliminate this essential enzyme. Symptoms begin in the first months of life, with feeding problems, poor weight gain, muscle weakness, floppiness, and head lag. Respiratory difficulties are often complicated by lung infections. Many infants with Pompe disease also have enlarged tongues. (Con

In [ ]:
# import pandas as pd
# import numpy as np
# from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline, AutoModelForSeq2SeqLM
# from sentence_transformers import SentenceTransformer
# import faiss
# import torch
# from fuzzywuzzy import process
# from fuzzywuzzy import fuzz
# import os

# # Load the dataset
# df = pd.read_csv('medquad.csv', encoding='latin-1')  # Change encoding if necessary
# df = df.groupby('question').agg({'answer': lambda x: ' '.join(x.astype(str))}).reset_index()
# medical_data = dict(zip(df['question'], df['answer']))

# # Model loading for NER, embeddings, and summarization
# model_name = "dmis-lab/biobert-v1.1"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForTokenClassification.from_pretrained(model_name)

# # Set NER pipeline to use CPU if GPU has issues
# device = "cpu"
# ner_pipeline = pipeline("ner", model=model, tokenizer=tokenizer, device=device)

# embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# # Generate or load embeddings with optimized FAISS indexing
# embedding_matrix_path = "embedding_matrix.npy"
# index_path = "faiss_index_hnsw.bin"

# if not os.path.exists(embedding_matrix_path):
#     embeddings = []
#     batch_size = 64
#     for i in range(0, len(df), batch_size):
#         batch = df['question'][i:i + batch_size].tolist()
#         batch_embeddings = embedder.encode(batch, convert_to_tensor=True, show_progress_bar=True)
#         embeddings.append(batch_embeddings)
#     embedding_matrix = torch.cat(embeddings).cpu().detach().numpy()
#     np.save(embedding_matrix_path, embedding_matrix)
# else:
#     embedding_matrix = np.load(embedding_matrix_path)

# faiss.normalize_L2(embedding_matrix)
# d = embedding_matrix.shape[1]

# # Use HNSW index for faster retrieval on large datasets
# if os.path.exists(index_path):
#     index = faiss.read_index(index_path)
# else:
#     index = faiss.IndexHNSWFlat(d, 32)  # Using HNSW index with 32 neighbors
#     index.add(embedding_matrix)
#     faiss.write_index(index, index_path)

# # Summarization model setup with CPU fallback
# summarizer_model_name = "facebook/bart-large-cnn"
# summarizer_tokenizer = AutoTokenizer.from_pretrained(summarizer_model_name)
# summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(summarizer_model_name)
# summarization_pipeline = pipeline("summarization", model=summarizer_model, tokenizer=summarizer_tokenizer, device=device)

# # Enhanced Abbreviation Dictionary
# abbreviations = {
#     "BP": "blood pressure",
#     "DM": "diabetes mellitus",
#     "P.A.D.": "peripheral arterial disease",
#     "HDL": "high-density lipoprotein",
#     "LDL": "low-density lipoprotein",
#     "HR": "heart rate",
#     "MRI": "magnetic resonance imaging"
#     # Additional entries can be added here
# }

# # Expand abbreviations in context
# def expand_abbreviations(text):
#     words = text.split()
#     expanded_words = [abbreviations.get(word, word) for word in words]
#     return ' '.join(expanded_words)

# df['question'] = df['question'].apply(expand_abbreviations)
# medical_data = dict(zip(df['question'], df['answer']))

# # Summarize text with enhanced length handling
# def summarize_text(text, min_length=100):
#     max_length = min(len(text) * 4 // 5, 500)
#     try:
#         summary = summarization_pipeline(
#             text, max_length=max_length, min_length=min_length, do_sample=False
#         )
#         return summary[0]['summary_text']
#     except Exception as e:
#         print(f"Summarization error: {e}")
#         return text[:max_length]

# # Improved entity extraction with error handling
# def extract_entities(text):
#     try:
#         tokens = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
#         outputs = model(**tokens)
#         predictions = torch.argmax(outputs.logits, dim=2)
#         entities = [tokenizer.decode([tok]) for pred, tok in zip(predictions[0], tokens['input_ids'][0]) if pred != 0]
#         return entities
#     except Exception as e:
#         print(f"Entity extraction error: {e}")
#         return []

# # Calculate Confidence Scores
# def calculate_confidence_scores(question_embedding, ranked_indices):
#     # Get embeddings of ranked answers
#     answer_embeddings = embedding_matrix[ranked_indices]

#     # Compute cosine similarities
#     cosine_similarities = np.dot(answer_embeddings, question_embedding.T).flatten()

#     # Normalize to a percentage scale (0-100)
#     confidence_scores = (cosine_similarities - cosine_similarities.min()) / (
#         cosine_similarities.max() - cosine_similarities.min()
#     ) * 100
#     return confidence_scores

# # Enhanced Answer Ranking with FAISS
# def rank_answers(question_embedding, top_k=5):
#     question_embedding = np.array(question_embedding)  # Ensure it's a numpy array
#     question_embedding = question_embedding.reshape(1, -1)  # Ensure it's 2D
#     faiss.normalize_L2(question_embedding)
#     _, indices = index.search(question_embedding, top_k) # top_k is now an integer, defaulting to 5
#     ranked_answers = [(df['answer'][i], i) for i in indices[0]]
#     return ranked_answers

# # Enhanced answer generation function with Confidence Scores
# def generate_answer(question):
#     question = expand_abbreviations(question)
#     question_embedding = embedder.encode([question], convert_to_tensor=True).cpu().detach().numpy()
#     ranked_answers = rank_answers(question_embedding) # Removed embedding_matrix argument

#     top_ranked_answers = []
#     for answer_text, idx in ranked_answers:
#         context = medical_data[df['question'][idx]]
#         summarized_answer = summarize_text(context)
#         top_ranked_answers.append((summarized_answer, idx))

#     # Calculate confidence scores
#     confidence_scores = calculate_confidence_scores(question_embedding, [idx for _, idx in ranked_answers])

#     if top_ranked_answers:
#         main_answer = top_ranked_answers[0][0]
#         main_confidence = confidence_scores[0]

#         related_answers_text = "\n".join(
#             [
#                 f"- {ans} (Confidence: {conf:.2f}%)"
#                 for ans, conf in zip([ans for ans, _ in top_ranked_answers][1:], confidence_scores[1:])
#             ]
#         )

#         return (
#             f"Answer: {main_answer} (Confidence: {main_confidence:.2f}%)\n\n"
#             f"Related answers:\n{related_answers_text}"
#         )
#     else:
#         return "Sorry, I couldn't find a relevant answer for your question."


